#### Citation

In [ ]:
# reference
# https://github.com/recommenders-team/recommenders
# https://github.com/recommenders-team/recommenders/blob/main/examples/00_quick_start/als_movielens.ipynb
# https://github.com/recommenders-team/recommenders/blob/main/examples/02_model_collaborative_filtering/als_deep_dive.ipynb

### Import Library

In [1]:
# 1. The Movies Dataset
!kaggle datasets download -d rounakbanik/the-movies-dataset
!unzip -q the-movies-dataset.zip -d movies_data

# 2. Book Recommendation Dataset
!kaggle datasets download -d arashnic/book-recommendation-dataset
!unzip -q book-recommendation-dataset.zip -d books_data

Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0
100% 228M/228M [00:01<00:00, 155MB/s]

Dataset URL: https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset
License(s): CC0-1.0
100% 24.3M/24.3M [00:00<00:00, 118MB/s]



In [2]:
!pip install pyspark
!pip install recommenders

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 7.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.3/355.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.6/303.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.6 MB/s eta 

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import pyspark
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.ml.tuning import CrossValidator
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import FloatType, IntegerType, LongType

from recommenders.utils.notebook_utils import is_jupyter
from recommenders.utils.timer import Timer
from recommenders.datasets import movielens
from recommenders.utils.spark_utils import start_or_get_spark
from recommenders.evaluation.spark_evaluation import SparkRankingEvaluation, SparkRatingEvaluation
from recommenders.tuning.parameter_sweep import generate_param_grid
from recommenders.datasets.spark_splitters import spark_random_split

print(f"System version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"PySpark version: {pyspark.__version__}")

System version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pandas version: 2.2.2
PySpark version: 4.0.2


### Data Preprocessing

In [5]:
# movie dataset loading
import pandas as pd

movies_data = 'movies_data/movies_metadata.csv'
ratings_data = 'movies_data/ratings_small.csv'
links_data = 'movies_data/links_small.csv'
movies_df = pd.read_csv(movies_data)
ratings_df = pd.read_csv(ratings_data)
links_df = pd.read_csv(links_data)

/tmp/ipykernel_22695/2310886920.py:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv(movies_data)


In [6]:
movies_df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


In [7]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [8]:
links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [9]:
# check shape
print('movies_df shape:', movies_df.shape)
print('ratings_df shape:', ratings_df.shape)
print('links_df shape', links_df.shape)

movies_df shape: (45466, 24)
ratings_df shape: (100004, 4)
links_df shape (9125, 3)


In [10]:
# check ratings missing values
ratings_df.isna().sum()

,0
userId,0
movieId,0
rating,0
timestamp,0


In [11]:
# check movies missing values
movies_df.isna().sum()

,0
adult,0
belongs_to_collection,40972
budget,0
genres,0
homepage,37684
id,0
imdb_id,17
original_language,11
original_title,0
overview,954


In [12]:
# check links missing values before dropping
print('Before dropping:\n', links_df.isna().sum())

# Drop rows where tmdbId is NaN
links_df = links_df.dropna(subset=['tmdbId'])

# Verify after dropping
print('\nAfter dropping:\n', links_df.isna().sum())

Before dropping:
 movieId     0
imdbId      0
tmdbId     13
dtype: int64

After dropping:
 movieId    0
imdbId     0
tmdbId     0
dtype: int64


In [13]:
# count unique userID in ratings_df
print(len(ratings_df["userId"].unique()))

671


In [14]:
# count unique movieID in ratings_df
print(len(ratings_df['movieId'].unique()))

9066


In [15]:
# count unique movieID in links_df
print(len(links_df['movieId'].unique()))

9112


In [16]:
# count unique movieID in links_df
print(len(links_df['tmdbId'].unique()))

9112


In [17]:
# count unique movieID in links_df
print(len(links_df['imdbId'].unique()))

9112


In [18]:
# default pandas setting
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')

In [ ]:
# modify pandas setting
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', None)

In [19]:
movies_df[['id','imdb_id','genres','title','overview']].head()

,id,imdb_id,genres,title,overview
0,862,tt0114709,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,8844,tt0113497,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Jumanji,When siblings Judy and Peter discover an encha...
2,15602,tt0113228,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",Grumpier Old Men,A family wedding reignites the ancient feud be...
3,31357,tt0114885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,11862,tt0113041,"[{'id': 35, 'name': 'Comedy'}]",Father of the Bride Part II,Just when George Banks has recovered from his ...


In [20]:
# filter unnecessary column
movie_filtered_df = movies_df[['id','genres','title','overview']].copy()

In [21]:
movie_filtered_df.head()

,id,genres,title,overview
0,862,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,8844,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Jumanji,When siblings Judy and Peter discover an encha...
2,15602,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",Grumpier Old Men,A family wedding reignites the ancient feud be...
3,31357,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,11862,"[{'id': 35, 'name': 'Comedy'}]",Father of the Bride Part II,Just when George Banks has recovered from his ...


In [22]:
# modify genres
import ast

# 1) convert string representation -> real list of dicts
movie_filtered_df['genres'] = movie_filtered_df['genres'].apply(ast.literal_eval)

# 2) store genres -> string format (seperate by | )
movie_filtered_df['genres'] = movie_filtered_df['genres'].apply(
    lambda g: "|".join(d["name"] for d in g) if isinstance(g, list) else ""
)

movie_filtered_df.head()

,id,genres,title,overview
0,862,Animation|Comedy|Family,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,8844,Adventure|Fantasy|Family,Jumanji,When siblings Judy and Peter discover an encha...
2,15602,Romance|Comedy,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,31357,Comedy|Drama|Romance,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,11862,Comedy,Father of the Bride Part II,Just when George Banks has recovered from his ...


In [23]:
# rename id -> tmdbId (matching)
movie_filtered_df = movie_filtered_df.rename(columns={'id': 'tmdbId'})

# Display the first few rows of the dataframe
movie_filtered_df.head()

,tmdbId,genres,title,overview
0,862,Animation|Comedy|Family,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,8844,Adventure|Fantasy|Family,Jumanji,When siblings Judy and Peter discover an encha...
2,15602,Romance|Comedy,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,31357,Comedy|Drama|Romance,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,11862,Comedy,Father of the Bride Part II,Just when George Banks has recovered from his ...


In [ ]:
# # Check for duplicate tmdbId
# duplicate_mask = movie_filtered_df.duplicated(subset=['tmdbId'], keep=False)
# duplicate_count = duplicate_mask.sum()
# print(f'Number of duplicate tmdbId entries: {duplicate_count}')

# # List the duplicated rows
# if duplicate_count > 0:
#     display(movie_filtered_df[duplicate_mask].sort_values('tmdbId'))

In [24]:
# Remove duplicate tmdbId entries, keeping only the first occurrence
movie_filtered_df = movie_filtered_df.drop_duplicates(subset=['tmdbId'], keep='first')

# Re-check for duplicate tmdbId
duplicate_count = movie_filtered_df.duplicated(subset=['tmdbId']).sum()
print(f'Number of duplicate tmdbId entries after removal: {duplicate_count}')

# Display the new shape to see how many rows remain
print(f'New movie_filtered_df shape: {movie_filtered_df.shape}')

Number of duplicate tmdbId entries after removal: 0
New movie_filtered_df shape: (45436, 4)


In [25]:
# check missing values again
movie_filtered_df.isna().sum()

,0
tmdbId,0
genres,0
title,6
overview,954


In [26]:
# store the genres in list format
movie_filtered_df['genres'] = movie_filtered_df['genres'].str.split('|')
movie_filtered_df.head()

,tmdbId,genres,title,overview
0,862,"[Animation, Comedy, Family]",Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,8844,"[Adventure, Fantasy, Family]",Jumanji,When siblings Judy and Peter discover an encha...
2,15602,"[Romance, Comedy]",Grumpier Old Men,A family wedding reignites the ancient feud be...
3,31357,"[Comedy, Drama, Romance]",Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,11862,[Comedy],Father of the Bride Part II,Just when George Banks has recovered from his ...


In [27]:
# filter links_df (only movieId, tmdbId)
links_filtered_df = links_df[['movieId', 'tmdbId']].copy()
links_filtered_df.head()

,movieId,tmdbId
0,1,862.0
1,2,8844.0
2,3,15602.0
3,4,31357.0
4,5,11862.0


In [28]:
movie_filtered_df.head()

,tmdbId,genres,title,overview
0,862,"[Animation, Comedy, Family]",Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,8844,"[Adventure, Fantasy, Family]",Jumanji,When siblings Judy and Peter discover an encha...
2,15602,"[Romance, Comedy]",Grumpier Old Men,A family wedding reignites the ancient feud be...
3,31357,"[Comedy, Drama, Romance]",Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,11862,[Comedy],Father of the Bride Part II,Just when George Banks has recovered from his ...


In [ ]:
# # movie_filtered_df data cleaning -> convert non-numeric values -> NaN
# count = pd.to_numeric(movie_filtered_df['tmdbId'], errors='coerce').isna().sum()
# print('invalid data count:', count)
# movie_filtered_df[pd.to_numeric(movie_filtered_df['tmdbId'], errors='coerce').isna()]['tmdbId'].head()

In [29]:
# movie_filtered_df data cleaning -> convert non-numeric values -> NaN
# pd.to_numeric with errors='coerce' turns non-numbers into NaN
movie_invalid_mask = pd.to_numeric(movie_filtered_df['tmdbId'], errors='coerce').isna()
count = movie_invalid_mask.sum()
print('invalid data count:', count)

# Show the first few invalid entries to understand the data issue
movie_filtered_df[movie_invalid_mask]['tmdbId'].head()

invalid data count: 3


,tmdbId
19730,1997-08-20
29503,2012-09-29
35587,2014-01-01


In [30]:
# links_filtered_df data cleaning -> convert non-numeric values -> NaN
count = pd.to_numeric(links_filtered_df['tmdbId'], errors='coerce').isna().sum()
print('invalid data count:', count)
links_filtered_df[pd.to_numeric(links_filtered_df['tmdbId'], errors='coerce').isna()]['tmdbId'].head()

invalid data count: 0


,tmdbId


In [31]:
# drop invalid data
movie_filtered_df['tmdbId'] = pd.to_numeric(movie_filtered_df['tmdbId'], errors='coerce')
links_filtered_df['tmdbId'] = pd.to_numeric(links_filtered_df['tmdbId'], errors='coerce')

movie_filtered_df = movie_filtered_df.dropna(subset=['tmdbId'])
links_filtered_df = links_filtered_df.dropna(subset=['tmdbId'])

In [32]:
# movie_filtered_df data cleaning -> convert non-numeric values -> NaN
count = pd.to_numeric(movie_filtered_df['tmdbId'], errors='coerce').isna().sum()
print('invalid data count:', count)
movie_filtered_df[pd.to_numeric(movie_filtered_df['tmdbId'], errors='coerce').isna()]['tmdbId'].head()

invalid data count: 0


,tmdbId


In [33]:
# links_filtered_df data cleaning -> convert non-numeric values -> NaN
count = pd.to_numeric(links_filtered_df['tmdbId'], errors='coerce').isna().sum()
print('invalid data count:', count)
links_filtered_df[pd.to_numeric(links_filtered_df['tmdbId'], errors='coerce').isna()]['tmdbId'].head()

invalid data count: 0


,tmdbId


In [34]:
# ensure same dtype

# avoid affect original data
movie_filtered_df = movie_filtered_df.copy()
links_filtered_df = links_filtered_df.copy()

movie_filtered_df['tmdbId'] = movie_filtered_df['tmdbId'].astype(int)
links_filtered_df['tmdbId'] = links_filtered_df['tmdbId'].astype(int)

# how='inner' ensures only rows with exactly matching 'tmdbId' in both dfs are kept
movie_merged_df = movie_filtered_df.merge(links_filtered_df, on='tmdbId', how='inner').copy()
movie_merged_df.head()

,tmdbId,genres,title,overview,movieId
0,862,"[Animation, Comedy, Family]",Toy Story,"Led by Woody, Andy's toys live happily in his ...",1
1,8844,"[Adventure, Fantasy, Family]",Jumanji,When siblings Judy and Peter discover an encha...,2
2,15602,"[Romance, Comedy]",Grumpier Old Men,A family wedding reignites the ancient feud be...,3
3,31357,"[Comedy, Drama, Romance]",Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",4
4,11862,[Comedy],Father of the Bride Part II,Just when George Banks has recovered from his ...,5


In [35]:
print(f"Total rows in merged dataframe: {len(movie_merged_df)}")

# Check for duplicate tmdbId
tmdb_dups = movie_merged_df.duplicated(subset=['tmdbId']).sum()
print(f"Duplicate tmdbId count: {tmdb_dups}")

# Check for duplicate movieId
movie_dups = movie_merged_df.duplicated(subset=['movieId']).sum()
print(f"Duplicate movieId count: {movie_dups}")

# If there are duplicates, show a few examples
if tmdb_dups > 0 or movie_dups > 0:
    print('\nExample of duplicated rows:')
    display(movie_merged_df[movie_merged_df.duplicated(subset=['tmdbId'], keep=False) |
                            movie_merged_df.duplicated(subset=['movieId'], keep=False)].sort_values('tmdbId'))

Total rows in merged dataframe: 9082
Duplicate tmdbId count: 0
Duplicate movieId count: 0


In [36]:
# copy movie_merged_df -> only movie & genres
movie_genres_df = movie_merged_df[['movieId', 'title', 'genres']].copy(deep=True)
movie_genres_df.head()

,movieId,title,genres
0,1,Toy Story,"[Animation, Comedy, Family]"
1,2,Jumanji,"[Adventure, Fantasy, Family]"
2,3,Grumpier Old Men,"[Romance, Comedy]"
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II,[Comedy]


### ALS Collaborative Filtering

**0. Load MovieLen Dataset**

In [37]:
ratings_df.head(5)

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [38]:
# top k items to recommend
TOP_K = 10

# Select MovieLens data size: 100k, 1m, 10m, or 20m
MOVIELENS_DATA_SIZE = '100k'

# Column names for the dataset
COL_USER = "userId"
COL_ITEM = "movieId"
COL_RATING = "rating"
COL_TIMESTAMP = "timestamp"

**1. Set Up Spark Context**

In [39]:
# the following settings work well for debugging locally on VM - change when running on a cluster
# set up a giant single executor with many threads and specify memory cap
spark = start_or_get_spark("ALS PySpark", memory="16g")
spark.conf.set("spark.sql.analyzer.failAmbiguousSelfJoin", "false")

**2. Split Data Train Test 80%, 20%**

In [40]:
ratings_spark_df = spark.createDataFrame(ratings_df)
train, test = ratings_spark_df.randomSplit([0.8, 0.2], seed=42)
print ("N train", train.cache().count())
print ("N test", test.cache().count())

N train 80066
N test 19938


In [41]:
train.show(5)
train.count()

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|     31|   2.5|1260759144|
|     1|   1029|   3.0|1260759179|
|     1|   1129|   2.0|1260759185|
|     1|   1172|   4.0|1260759205|
|     1|   1263|   2.0|1260759151|
+------+-------+------+----------+
only showing top 5 rows


80066

In [42]:
test.show(5)
test.count()

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|   1061|   3.0|1260759182|
|     1|   1287|   2.0|1260759187|
|     1|   1339|   3.5|1260759125|
|     1|   2105|   4.0|1260759139|
|     1|   3671|   3.0|1260759117|
+------+-------+------+----------+
only showing top 5 rows


19938

**3. Train ALS model (train data) > Get top-k recommendations (test data)**

In [43]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

header = {
    "userCol": COL_USER,
    "itemCol": COL_ITEM,
    "ratingCol": COL_RATING,
}

als = ALS(
    rank=10,
    maxIter=20,
    regParam=0.1,
    implicitPrefs=False,
    coldStartStrategy='drop',
    nonnegative=False,
    seed=42,
    **header
)

# # Define the hyperparameter grid
# param_grid = ParamGridBuilder() \
#     .addGrid(als.rank, [10, 20, 50]) \
#     .addGrid(als.regParam, [0.01, 0.05, 0.1]) \
#     .addGrid(als.maxIter, [10, 20]) \
#     .build()

# # Define the evaluator
# evaluator = RegressionEvaluator(
#     metricName="rmse",
#     labelCol=COL_RATING,
#     predictionCol="prediction"
# )

# # Set up CrossValidator
# cv = CrossValidator(
#     estimator=als,
#     estimatorParamMaps=param_grid,
#     evaluator=evaluator,
#     numFolds=3 # 3-fold cross validation
# )

print("CrossValidator setup complete. You can now fit `cv` on your training data (e.g., `model = cv.fit(train)`).")

CrossValidator setup complete. You can now fit `cv` on your training data (e.g., `model = cv.fit(train)`).


Load Model or Train Model(if not exist known model)

In [44]:
import os
import zipfile
from pyspark.ml.recommendation import ALSModel

zip_path = "als_model_001.zip"
model_dir = "als_model"

if os.path.exists(zip_path):
    print(f"Found {zip_path}, loading the model...")
    if not os.path.exists(model_dir):
        print("Extracting model files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall()
    model = ALSModel.load(model_dir)
    print(f"Model successfully loaded from {model_dir}")
else:
    print(f"{zip_path} not found. Training a new model...")
    with Timer() as train_time:
        model = als.fit(train)
    print("Took {} seconds for training.".format(train_time.interval))

als_model_001.zip not found. Training a new model...
Took 31.514327790999232 seconds for training.


In [ ]:
# with Timer() as train_time:
#     model = cv.fit(train)

# print("Took {} seconds for training.".format(train_time.interval))

##### Load Model

In [ ]:
# # Save the trained PySpark ALS model
# model_path = "als_model"

# # Using write().overwrite().save() to save the model and overwrite if it already exists
# model.write().overwrite().save(model_path)

# print(f"Model successfully saved to {model_path}")

Model successfully saved to als_model


In [ ]:
# # 1. Compress the folder on your local machine into a .zip file
# # 2. Use the "Files" tab on the left sidebar to upload the .zip file
# # 3. Use this code in a cell to unzip it
# !unzip als_model_001.zip


Archive:  als_model_001.zip
  inflating: als_model/metadata/_SUCCESS  
  inflating: als_model/userFactors/_SUCCESS  
  inflating: als_model/itemFactors/_SUCCESS  
  inflating: als_model/userFactors/part-00001-864e5927-4f09-41cb-9aaa-886dda65ad28-c000.snappy.parquet  
  inflating: als_model/userFactors/part-00006-864e5927-4f09-41cb-9aaa-886dda65ad28-c000.snappy.parquet  
  inflating: als_model/userFactors/.part-00005-864e5927-4f09-41cb-9aaa-886dda65ad28-c000.snappy.parquet.crc  
  inflating: als_model/userFactors/part-00009-864e5927-4f09-41cb-9aaa-886dda65ad28-c000.snappy.parquet  
  inflating: als_model/userFactors/.part-00007-864e5927-4f09-41cb-9aaa-886dda65ad28-c000.snappy.parquet.crc  
  inflating: als_model/userFactors/part-00007-864e5927-4f09-41cb-9aaa-886dda65ad28-c000.snappy.parquet  
  inflating: als_model/metadata/.part-00000-a83d7046-07df-4d65-9eb9-f4a6c3cc418c-c000.txt.crc  
  inflating: als_model/userFactors/.part-00006-864e5927-4f09-41cb-9aaa-886dda65ad28-c000.snappy.parqu

In [ ]:
# from pyspark.ml.recommendation import ALSModel

# # Load the saved PySpark ALS model
# model_path = "als_model"
# loaded_model = ALSModel.load(model_path)
# model = loaded_model

# print(f"Model successfully loaded from {model_path}")

Model successfully loaded from als_model


##### Testing Recommendations

In [45]:
# Create a DataFrame with the user(s) you want to generate recommendations for
user_subset = spark.createDataFrame([(1,)], ["userId"])

# Generate top 5 movie recommendations for the specified user(s)
recommendations = model.recommendForUserSubset(user_subset, 5)

# Display the recommendations
recommendations.show(truncate=False)

+------+----------------------------------------------------------------------------------------------+
|userId|recommendations                                                                               |
+------+----------------------------------------------------------------------------------------------+
|1     |[{1192, 3.853778}, {5047, 3.8194106}, {45666, 3.7672148}, {1635, 3.7641108}, {1150, 3.687649}]|
+------+----------------------------------------------------------------------------------------------+



In [46]:
# Generate predicted ratings for the test dataset using the loaded model
prediction = model.transform(test)

# Show the first few prediction results
print("Predictions using the loaded model:")
prediction.show(5)

Predictions using the loaded model:
+------+-------+------+----------+----------+
|userId|movieId|rating| timestamp|prediction|
+------+-------+------+----------+----------+
|   148|    185|   3.0|1059604099| 3.0772088|
|   148|    364|   4.0|1059604221| 4.1017413|
|   148|    596|   4.5|1059604241| 3.9236002|
|   148|   1028|   5.0|1059505000| 4.0014405|
|   148|   1136|   4.5|1059604684| 4.4160366|
+------+-------+------+----------+----------+
only showing top 5 rows


##### Evaluation

In the movie recommendation use case, recommending movies that have been rated by the users do not make sense. Therefore, the rated movies are removed from the recommended items.

In order to achieve this, we recommend all movies to all users, and then remove the user-movie pairs that exist in the training dataset.

In [47]:
with Timer() as test_time:

    # Get the cross join of all user-item pairs and score them.
    users = train.select(COL_USER).distinct()
    items = train.select(COL_ITEM).distinct()
    user_item = users.crossJoin(items)
    dfs_pred = model.transform(user_item)

    # Remove seen items.
    dfs_pred_exclude_train = dfs_pred.alias("pred").join(
        train.alias("train"),
        (dfs_pred[COL_USER] == train[COL_USER]) & (dfs_pred[COL_ITEM] == train[COL_ITEM]),
        how='outer'
    )

    top_all = dfs_pred_exclude_train.filter(dfs_pred_exclude_train[f"train.{COL_RATING}"].isNull()) \
        .select('pred.' + COL_USER, 'pred.' + COL_ITEM, 'pred.' + "prediction")

    # In Spark, transformations are lazy evaluation
    # Use an action to force execute and measure the test time
    top_all.cache().count()

print("Took {} seconds for prediction.".format(test_time.interval))

Took 51.756820698999945 seconds for prediction.


In [48]:
top_all.show()

+------+-------+----------+
|userId|movieId|prediction|
+------+-------+----------+
|     1|     94| 1.6520252|
|     1|    185| 1.4515272|
|     1|    328| 1.9737031|
|     1|    354| 2.2093885|
|     1|    367| 2.1360717|
|     1|   1086| 2.1644778|
|     1|   1500| 2.2914195|
|     1|   1620| 1.5648633|
|     1|   2393| 1.4143732|
|     1|   2597| 0.9665849|
|     1|   2857| 2.6316445|
|     1|   2868|   0.91016|
|     1|   2984| 2.8593261|
|     1|   3134|  2.780806|
|     1|   3437| 2.7948182|
|     1|   3462| 2.9112363|
|     1|   4007|  2.572688|
|     1|   5025|0.97208333|
|     1|   5203| 1.6894495|
|     1|   5342| 0.8447248|
+------+-------+----------+
only showing top 20 rows


**4. Evaluate ALS**

In [49]:
rank_eval = SparkRankingEvaluation(test, top_all, k = TOP_K, col_user=COL_USER, col_item=COL_ITEM,
                                    col_rating=COL_RATING, col_prediction="prediction",
                                    relevancy_method="top_k")

In [50]:
print("Model:\tALS",
      "Top K:\t%d" % rank_eval.k,
      "MAP:\t%f" % rank_eval.map_at_k(),
      "NDCG:\t%f" % rank_eval.ndcg_at_k(),
      "Precision@K:\t%f" % rank_eval.precision_at_k(),
      "Recall@K:\t%f" % rank_eval.recall_at_k(), sep='\n')

# print("\n--- How to improve these ranking metrics? ---")
# print("1. Hyperparameter Tuning: Use CrossValidator and ParamGridBuilder to find the best 'rank', 'regParam', and 'maxIter'.")
# print("2. Implicit Feedback: If using implicit data, set implicitPrefs=True in the ALS model.")
# print("3. More Data: Train on larger MovieLens datasets (e.g., 1M, 10M, 20M) to reduce sparsity.")
# print("4. Hybrid Models: Incorporate user demographics or movie genres (which you preprocessed earlier) using hybrid recommenders.")

Model:	ALS
Top K:	10
MAP:	0.000339
NDCG:	0.001363
Precision@K:	0.001639
Recall@K:	0.000229


**5. Evaluate rating prediction**

In [51]:
# Generate predicted ratings.
prediction = model.transform(test)
prediction.cache().show()

+------+-------+------+----------+----------+
|userId|movieId|rating| timestamp|prediction|
+------+-------+------+----------+----------+
|   148|    185|   3.0|1059604099| 3.0772088|
|   148|    364|   4.0|1059604221| 4.1017413|
|   148|    596|   4.5|1059604241| 3.9236002|
|   148|   1028|   5.0|1059505000| 4.0014405|
|   148|   1136|   4.5|1059604684| 4.4160366|
|   148|   1172|   5.0|1059680324|  4.391507|
|   148|   1208|   5.0|1059504953| 4.2399797|
|   148|   1247|   4.0|1059504938| 3.9893665|
|   148|   1517|   4.5|1059504984|  3.048558|
|   148|   1625|   4.0|1059604479| 3.8985784|
|   148|   2053|   2.5|1059604160| 1.9999485|
|   148|   2116|   4.0|1059531240| 3.2932887|
|   148|   2628|   2.5|1059604051| 3.4971304|
|   148|   2986|   3.5|1059530938| 2.5728507|
|   148|   3897|   4.5|1059604694| 4.2915545|
|   148|   4734|   2.0|1059680407|  3.357071|
|   148|   4835|   4.0|1059507563|  3.783707|
|   148|   5944|   3.5|1059530891| 3.1343372|
|   148|   5945|   4.5|1059507529|

In [52]:
rating_eval = SparkRatingEvaluation(test, prediction, col_user=COL_USER, col_item=COL_ITEM,
                                    col_rating=COL_RATING, col_prediction="prediction")

print("Model:\tALS rating prediction",
      "RMSE:\t%f" % rating_eval.rmse(),
      "MAE:\t%f" % rating_eval.mae(),
      "Explained variance:\t%f" % rating_eval.exp_var(),
      "R squared:\t%f" % rating_eval.rsquared(), sep='\n')

Model:	ALS rating prediction
RMSE:	0.912891
MAE:	0.708384
Explained variance:	0.275674
R squared:	0.248871


#### End Spark

In [ ]:
# cleanup spark instance
# spark.stop()

### LLM

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 13.6 gigabytes of available RAM

Not using a high-RAM runtime


In [ ]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.0-flash',
 'google/gemini-2.0-flash-lite',
 'google/gemini-2.5-flash',
 'google/gemini-2.5-flash-lite',
 'google/gemini-2.5-pro',
 'google/gemini-3-pro-preview',
 'google/gemma-3-12b',
 'google/gemma-3-1b',
 'google/gemma-3-27b',
 'google/gemma-3-4b']

In [ ]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

The capital of France is **Paris**.


In [ ]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-3-pro-preview')
print(response)

The capital of England is **London**.


In [ ]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

Elara was a girl made of quiet moments. She preferred the rustle of leaves to the clamor of the village, and the company of ancient trees to the chatter of her peers. She often felt small, a tiny pebble on a vast beach.

One sun-dappled afternoon, she wandered deeper than usual into the old woods bordering her home. She stumbled upon a hidden glade, a circular clearing bathed in soft light. In its center stood a willow tree, unlike any she had ever seen. Its branches, impossibly long and trailing, touched the ground, forming a verdant, living curtain around its trunk. It looked like a sorrowful queen, draped in emerald tears.

Intrigued, Elara pushed aside the heavy strands and stepped into the hollow within. The air inside was cool and smelled of earth and damp leaves. A hushed silence enveloped her, broken only by the gentle *swish-swish-swish* of the leaves outside.

She sat at the base of the gnarled trunk, leaning against its rough bark. As she closed her eyes, the gentle swaying 